# 📚 Google Scholar Profile Scraper

Extract **everything** from any Google Scholar profile:
- 👤 Author bio, affiliation, interests
- 📊 h-index, i10-index, citations (all-time & 5-year)
- 📅 Year-by-year citation chart
- 🤝 Co-authors list
- 📄 All publications with full metadata
- 💾 Download results as **JSON** and **CSV**

---
> **How to use:** Run each cell top-to-bottom.  
> Only **Cell 2 (Configuration)** needs your input — paste the Scholar profile URL there.

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Install dependencies
# ─────────────────────────────────────────────────────────
!pip install scholarly --quiet
print('✅  scholarly installed successfully')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — ✏️  CONFIGURATION  (edit this cell)
# ─────────────────────────────────────────────────────────

# Paste the full Google Scholar profile URL  -OR-  just the user ID
# Example URL : "https://scholar.google.com/citations?user=JicYPdAAAAAJ"
# Example ID  : "JicYPdAAAAAJ"
PROFILE_URL = "https://scholar.google.com/citations?user=JicYPdAAAAAJ"  # ← change me

# Fetch full details for every publication?
# True  = complete data, slower (adds ~0.3s per paper)
# False = basic listing only, much faster
FILL_ALL_PUBLICATIONS = True

# (Optional) ScraperAPI key — helps if Google blocks the Colab IP
# Get a free key at https://www.scraperapi.com/
# Leave as None to run without a proxy
SCRAPER_API_KEY = None   # e.g. "abc123yourkeyhere"

print('⚙️  Configuration saved.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3 — Imports & helpers
# ─────────────────────────────────────────────────────────
import re, json, csv, time, pathlib, io
from datetime import datetime
from IPython.display import display, HTML
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scholarly import scholarly, ProxyGenerator

def extract_user_id(raw: str) -> str:
    match = re.search(r"user=([A-Za-z0-9_-]+)", raw)
    return match.group(1) if match else raw.strip().rstrip("/")

def safe_get(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict):
            return default
        d = d.get(k, default)
    return d

def setup_proxy(key):
    if key:
        pg = ProxyGenerator()
        pg.ScraperAPI(key)
        scholarly.use_proxy(pg)
        print('🔒  ScraperAPI proxy enabled.')
    else:
        print('ℹ️  No proxy configured — running direct (may hit rate-limits on large profiles).')

print('✅  Helpers loaded.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Fetch profile
# ─────────────────────────────────────────────────────────
setup_proxy(SCRAPER_API_KEY)

user_id = extract_user_id(PROFILE_URL)
print(f'\n🔍  Fetching profile for user ID: {user_id}')

author_raw = scholarly.search_author_id(user_id)
print('📥  Loading full profile sections…')
author = scholarly.fill(author_raw, sections=["basics", "indices", "counts", "coauthors", "publications"])

if FILL_ALL_PUBLICATIONS:
    pubs = author.get('publications', [])
    total = len(pubs)
    print(f'📄  Found {total} publications — fetching details…')
    for i, pub in enumerate(pubs, 1):
        try:
            scholarly.fill(pub)
            if i % 10 == 0 or i == total:
                print(f'   Progress: {i}/{total}', end='\r')
            time.sleep(0.3)
        except Exception as e:
            print(f'   ⚠  [{i}/{total}] skipped: {e}')
    print(f'\n✅  All {total} publications fetched.')
else:
    print(f'ℹ️  Skipping per-publication fill (FILL_ALL_PUBLICATIONS=False).')

print('\n🎉  Profile fetch complete!')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Profile summary card
# ─────────────────────────────────────────────────────────
bib      = author.get('bib', {})
name     = bib.get('name', 'Unknown')
aff      = bib.get('affiliation', 'N/A')
email    = bib.get('email', 'N/A')
interests= bib.get('interests', [])
url      = f"https://scholar.google.com/citations?user={user_id}"

cited_by  = author.get('citedby',    'N/A')
cited_by5 = author.get('citedby5y',  'N/A')
h         = author.get('hindex',     'N/A')
h5        = author.get('hindex5y',   'N/A')
i10       = author.get('i10index',   'N/A')
i10_5y    = author.get('i10index5y', 'N/A')
total_pubs= len(author.get('publications', []))
coauth_n  = len(author.get('coauthors', []))

card = f"""
<div style="font-family:sans-serif;border:1px solid #ddd;border-radius:12px;
            padding:20px 28px;max-width:680px;background:#f9f9ff;">
  <h2 style="margin:0 0 4px;color:#1a0dab">📖 {name}</h2>
  <p style="margin:0 0 12px;color:#555">{aff}</p>
  {'<p style="margin:0 0 12px;color:#555">✉️ ' + email + '</p>' if email != 'N/A' else ''}
  {'<p style="margin:0 0 16px">🏷️ ' + ' &nbsp;|&nbsp; '.join(f'<span style="background:#e8f0fe;padding:2px 8px;border-radius:20px;font-size:0.88em">{i}</span>' for i in interests) + '</p>' if interests else ''}
  <a href="{url}" target="_blank" style="color:#1a0dab;font-size:0.85em">🔗 View on Google Scholar</a>

  <hr style="margin:16px 0;border:none;border-top:1px solid #ddd">

  <table style="border-collapse:collapse;width:100%">
    <tr style="background:#e8eaf6">
      <th style="padding:8px 12px;text-align:left">Metric</th>
      <th style="padding:8px 12px;text-align:center">All-time</th>
      <th style="padding:8px 12px;text-align:center">Since {datetime.now().year-5}</th>
    </tr>
    <tr>
      <td style="padding:8px 12px">📖 Total Publications</td>
      <td style="padding:8px 12px;text-align:center">{total_pubs}</td>
      <td style="padding:8px 12px;text-align:center">—</td>
    </tr>
    <tr style="background:#f5f5f5">
      <td style="padding:8px 12px">📣 Citations</td>
      <td style="padding:8px 12px;text-align:center"><b>{cited_by}</b></td>
      <td style="padding:8px 12px;text-align:center">{cited_by5}</td>
    </tr>
    <tr>
      <td style="padding:8px 12px">🏆 h-index</td>
      <td style="padding:8px 12px;text-align:center"><b>{h}</b></td>
      <td style="padding:8px 12px;text-align:center">{h5}</td>
    </tr>
    <tr style="background:#f5f5f5">
      <td style="padding:8px 12px">🔟 i10-index</td>
      <td style="padding:8px 12px;text-align:center"><b>{i10}</b></td>
      <td style="padding:8px 12px;text-align:center">{i10_5y}</td>
    </tr>
    <tr>
      <td style="padding:8px 12px">🤝 Co-authors</td>
      <td style="padding:8px 12px;text-align:center">{coauth_n}</td>
      <td style="padding:8px 12px;text-align:center">—</td>
    </tr>
  </table>
</div>
"""
display(HTML(card))

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6 — Citations-per-year bar chart
# ─────────────────────────────────────────────────────────
cites_per_year = author.get('cites_per_year', {})

if cites_per_year:
    years  = sorted(cites_per_year.keys())
    counts = [cites_per_year[y] for y in years]

    fig, ax = plt.subplots(figsize=(12, 4))
    bars = ax.bar(years, counts, color='#4285F4', edgecolor='white', linewidth=0.6)

    # Annotate bars
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                str(count), ha='center', va='bottom', fontsize=8, color='#333')

    ax.set_title(f'Citations per Year — {name}', fontsize=14, fontweight='bold', pad=14)
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Citations', fontsize=11)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.spines[['top','right']].set_visible(False)
    ax.set_facecolor('#fafafa')
    fig.tight_layout()
    plt.show()
else:
    print('ℹ️  No citations-per-year data available for this profile.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7 — Publications table (interactive DataFrame)
# ─────────────────────────────────────────────────────────
publications = author.get('publications', [])

rows = []
for pub in publications:
    b = pub.get('bib', {})
    rows.append({
        'Title'      : b.get('title', ''),
        'Authors'    : b.get('author', ''),
        'Year'       : b.get('pub_year', ''),
        'Venue'      : b.get('venue', b.get('journal', '')),
        'Journal'    : b.get('journal', ''),
        'Volume'     : b.get('volume', ''),
        'Pages'      : b.get('pages', ''),
        'Publisher'  : b.get('publisher', ''),
        'Citations'  : pub.get('num_citations', 0),
        'Scholar URL': pub.get('pub_url', ''),
        'Abstract'   : b.get('abstract', ''),
    })

df_pubs = pd.DataFrame(rows).sort_values('Citations', ascending=False).reset_index(drop=True)
df_pubs.index += 1  # 1-based rank

print(f'📄  {len(df_pubs)} publications — sorted by citation count (descending)\n')

# Show a styled preview (first 50 rows)
display_cols = ['Title', 'Year', 'Venue', 'Citations']
styled = (
    df_pubs[display_cols].head(50)
    .style
    .background_gradient(subset=['Citations'], cmap='Blues')
    .set_properties(**{'font-size': '12px'})
    .set_table_styles([{'selector': 'th', 'props': [('background-color', '#4285F4'),
                                                     ('color', 'white'),
                                                     ('font-size', '13px')]}])
)
display(styled)

if len(df_pubs) > 50:
    print(f'\n… {len(df_pubs)-50} more rows in the exported CSV.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8 — Top 15 most-cited papers (horizontal bar chart)
# ─────────────────────────────────────────────────────────
top15 = df_pubs.head(15).copy()
top15['Short Title'] = top15['Title'].str[:55] + '…'

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.Blues_r([i/15 for i in range(15)])
bars = ax.barh(top15['Short Title'][::-1], top15['Citations'][::-1],
               color=colors, edgecolor='white')

for bar, val in zip(bars, top15['Citations'][::-1]):
    ax.text(bar.get_width() + max(top15['Citations'])*0.005,
            bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9, color='#333')

ax.set_title(f'Top 15 Most-Cited Papers — {name}', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Citations', fontsize=11)
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('#fafafa')
fig.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9 — Co-authors table
# ─────────────────────────────────────────────────────────
coauthors = author.get('coauthors', [])

if coauthors:
    ca_rows = []
    for ca in coauthors:
        ca_rows.append({
            'Name'        : safe_get(ca, 'bib', 'name', default=''),
            'Affiliation' : safe_get(ca, 'bib', 'affiliation', default=''),
            'Scholar ID'  : ca.get('scholar_id', ''),
        })
    df_coauth = pd.DataFrame(ca_rows)
    print(f'🤝  {len(df_coauth)} co-authors:\n')
    display(df_coauth.style.set_properties(**{'font-size': '12px'})
            .set_table_styles([{'selector': 'th',
                                'props': [('background-color','#34A853'),
                                          ('color','white'),
                                          ('font-size','13px')]}]))
else:
    print('ℹ️  No co-author data available.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 10 — Export & download files
# ─────────────────────────────────────────────────────────
from google.colab import files as colab_files

name_slug = re.sub(r'[^\w]+', '_', name).strip('_').lower()
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
base      = f"{name_slug}_{timestamp}"

# ── CSV ──────────────────────────────────────────────────
csv_path = f"{base}_articles.csv"
df_pubs.to_csv(csv_path, index_label='rank', encoding='utf-8')
print(f'💾  CSV written: {csv_path}')

# ── JSON ─────────────────────────────────────────────────
json_path = f"{base}_scholar.json"
def json_default(obj):
    if hasattr(obj, '__dict__'):
        return obj.__dict__
    return str(obj)

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(author, f, indent=2, ensure_ascii=False, default=json_default)
print(f'💾  JSON written: {json_path}')

# ── Download both ─────────────────────────────────────────
print('\n⬇️  Starting downloads…')
colab_files.download(csv_path)
colab_files.download(json_path)
print('✅  Done!')

---
## 💡 Tips & Troubleshooting

| Issue | Fix |
|---|---|
| **`MaxTriesExceededException`** | Google has rate-limited the Colab IP. Set `SCRAPER_API_KEY` in Cell 2 or wait ~15 min and retry. |
| **Empty profile returned** | Double-check the `user=` value in your URL. It is case-sensitive. |
| **Slow on large profiles** | Set `FILL_ALL_PUBLICATIONS = False` in Cell 2 for a fast overview. |
| **Want to save to Google Drive** | Replace `colab_files.download(...)` in Cell 10 with `shutil.copy(path, '/content/drive/MyDrive/')` after mounting Drive. |

### How to find the Scholar User ID
Open any Google Scholar profile → look at the URL:  
`https://scholar.google.com/citations?user=`**`XXXXXXXX`**`&hl=en`  
The bold part is the user ID.